# 04 - Data Augmentation

**Phase 1, Step 1.4 - Data Augmentation**

Defines and visualizes the augmentation pipeline used by every classifier training notebook: random horizontal flip, rotation (+/-15 degrees), brightness adjustment (+/-20%), contrast adjustment, random zoom, and color jittering. This notebook is a visual sanity check - the actual augmentation layer is re-declared inside each training notebook so they stay fully self-contained.

### 1. Mount Drive and load configuration

In [ ]:
!pip install -q datasets pyyaml tqdm scikit-learn matplotlib pandas opencv-python-headless tensorflow pillow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

# ---------------------------------------------------------------------------
# Project configuration - shared across every SmartVision AI notebook.
# All notebooks read/write under this same Google Drive folder so that
# work done in one notebook (e.g. dataset collection) is available to the
# next one (e.g. training), even across separate Colab sessions.
# ---------------------------------------------------------------------------
BASE_DIR = "/content/drive/MyDrive/SmartVisionAI"

RAW_DATA_DIR = os.path.join(BASE_DIR, "raw_data")
RAW_IMAGES_DIR = os.path.join(RAW_DATA_DIR, "images")
RAW_ANNOTATIONS_PATH = os.path.join(RAW_DATA_DIR, "annotations.json")

CLASSIFICATION_DIR = os.path.join(BASE_DIR, "classification")
CLASSIFICATION_TRAIN_DIR = os.path.join(CLASSIFICATION_DIR, "train")
CLASSIFICATION_VAL_DIR = os.path.join(CLASSIFICATION_DIR, "val")
CLASSIFICATION_TEST_DIR = os.path.join(CLASSIFICATION_DIR, "test")

DETECTION_DIR = os.path.join(BASE_DIR, "detection")
DETECTION_IMAGES_DIR = os.path.join(DETECTION_DIR, "images")
DETECTION_LABELS_DIR = os.path.join(DETECTION_DIR, "labels")
DETECTION_YAML_PATH = os.path.join(DETECTION_DIR, "data.yaml")

MODELS_DIR = os.path.join(BASE_DIR, "models")
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")

for d in [BASE_DIR, RAW_DATA_DIR, RAW_IMAGES_DIR, CLASSIFICATION_DIR, DETECTION_DIR, MODELS_DIR, OUTPUTS_DIR]:
    os.makedirs(d, exist_ok=True)

# The 25 selected COCO classes (must match COCO category names exactly)
SELECTED_CLASSES = [
    # Vehicles (6)
    "car", "truck", "bus", "motorcycle", "bicycle", "airplane",
    # Person (1)
    "person",
    # Outdoor (3)
    "traffic light", "stop sign", "bench",
    # Animals (6)
    "dog", "cat", "horse", "bird", "cow", "elephant",
    # Kitchen & food (5)
    "bottle", "cup", "bowl", "pizza", "cake",
    # Furniture & indoor (4)
    "chair", "couch", "bed", "potted plant",
]
assert len(SELECTED_CLASSES) == 25

CLASS_TO_IDX = {name: i for i, name in enumerate(SELECTED_CLASSES)}
IDX_TO_CLASS = {i: name for i, name in enumerate(SELECTED_CLASSES)}

def safe_name(class_name):
    return class_name.replace(" ", "_")

IMAGES_PER_CLASS = 350        # -> 8,750 images total (up from 100/class to fight overfitting)
TRAIN_SPLIT = 0.70
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

CLS_IMG_SIZE = 224            # Classification input resolution (single-resolution throughout)
FINE_TUNE_IMG_SIZE = 384      # Unused by classifier training (reverted to single-resolution); kept for compatibility
YOLO_IMG_SIZE = 640
BATCH_SIZE = 32                # Stage 1 batch size
BATCH_SIZE_STAGE2 = 16         # Smaller batch at 384x384 to fit GPU memory (~2.9x pixels/image)

HF_DATASET_NAME = "detection-datasets/coco"

print("BASE_DIR:", BASE_DIR)
print("Classes:", len(SELECTED_CLASSES))


### 2. Define the augmentation pipeline

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

augmentation_pipeline = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.04),     # ~ +/-15 degrees
    layers.RandomBrightness(0.2),    # +/-20%
    layers.RandomContrast(0.2),
    layers.RandomZoom(0.1),
], name="augmentation_pipeline")

print(augmentation_pipeline.summary())


### 3. Visualize augmented versions of a sample image

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

sample_class = safe_name(SELECTED_CLASSES[0])
files = glob.glob(os.path.join(CLASSIFICATION_TRAIN_DIR, sample_class, "*.jpg"))
assert files, "Run 03_Data_Preprocessing_Classification.ipynb first."

img = np.array(Image.open(files[0]).convert("RGB")).astype("float32")
batch = np.expand_dims(img, 0)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes[0, 0].imshow(img.astype("uint8"))
axes[0, 0].set_title("Original")
axes[0, 0].axis("off")

for ax in axes.flat[1:]:
    augmented = augmentation_pipeline(batch, training=True)[0].numpy()
    augmented = np.clip(augmented, 0, 255).astype("uint8")
    ax.imshow(augmented)
    ax.set_title("Augmented")
    ax.axis("off")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, "augmentation_examples.png"), dpi=150)
plt.show()


**Next notebooks:** `05_Train_VGG16.ipynb`, `06_Train_ResNet50.ipynb`, `07_Train_MobileNetV2.ipynb`, `08_Train_EfficientNetB0.ipynb` - each trains one classifier independently, reusing this same augmentation pipeline definition inline.